# Data Manipulation — Advanced
## Feature Engineering (R / tidymodels)

**Philosophy:** This note starts where cleaning and reshaping end — data is
clean and in the right shape. Each section covers how to create new features
for ML/modeling, with explicit attention to *when to use each technique* and
*what can go wrong (especially leakage)*.

**A structural note on this port:** the Python source leans heavily on
scikit-learn's `Pipeline` and `ColumnTransformer` — objects that bundle a
sequence of column-wise transforms into one fit-on-train-only unit. R's
**tidymodels** ecosystem solves the identical problem with a different
object model: a single `recipes::recipe()` accumulates `step_*()` calls (one
per transformation), and `prep()`/`bake()` (or `juice()`) replace
`fit()`/`transform()`. There is no direct `ColumnTransformer` equivalent to
transliterate line-by-line — instead, **each `step_*()` call already targets
specific columns via `tidyselect` semantics** (`all_numeric()`,
`all_nominal()`, explicit names), which is what `ColumnTransformer` exists to
bolt onto sklearn's column-blind transformers. This section is therefore a
genuine redesign, not a mechanical translation — see §8 for the full
`recipe()` capstone, the direct structural analog to this note's closing
`ColumnTransformer` example.

---

## Decision Table

| If your column is... | Go to |
| :--- | :--- |
| Categorical / string that needs to be numeric | §1 — Encoding Categoricals |
| Numeric but skewed, on different scales, or needs normalization | §2 — Numerical Transformations |
| Continuous numeric that should be grouped into ranges | §3 — Binning & Discretization |
| A datetime that should produce predictive features | §4 — Date & Time Features |
| A time-ordered numeric where past values predict future | §5 — Lag & Window Features |
| Two or more columns whose combination may be informative | §6 — Interaction & Ratio Features |
| Free text in a tabular dataset | §7 — Text as Features |
| Suspiciously good model performance or target in features | §8 — Leakage & Validation |

## Series Map

| Notebook | Use it for |
| :--- | :--- |
| `DM_Basic` | Cleaning & QC — audit, types, nulls, dupes, strings, outliers, column names |
| `DM_Intermediate` | Reshaping & combining — wide/long, group-by, multi-table joins, time-series, nested, many files |
| `DM_Advanced` **(this note)** | Feature engineering — encoding, transforms, binning, date, lag/window, interactions, text, leakage |
| `Reference_Tidyverse / Reference_BaseR` | Syntax lookups for individual functions |

> **Environment:** R 4.x / tidyverse 2.x / tidymodels 1.x. **Convention:**
> `X_train`/`X_test` are feature frames and `y_train` the target; in
> tidymodels idiom this is more often a single `train`/`test` split from
> `rsample`, with the target column kept alongside the features inside the
> `recipe()` — fit (`prep()`) every recipe on train only, then `bake()` test.
> This notebook's cells are reference-style (they assume your modelling
> data); the added cells below are self-contained and runnable.

---
## §1 — Encoding Categorical Variables

Most models require numeric input. The right encoding depends on cardinality,
whether order exists, and whether you're using a tree-based or linear model.

In [ ]:
library(dplyr)
library(recipes)

# ── Encoding 1: One-Hot Encoding ─────────────────────────────────────────────
# Use when: nominal (no order), low cardinality (<15 unique), linear models or NNs
# recipes' step_dummy() IS the get_dummies(drop_first=True) default -- it drops
# one level automatically (avoids the dummy variable trap) unless you ask it not to
rec <- recipe(~ platform, data = df) |> step_dummy(platform)
# For tree-based models, one_hot = TRUE keeps all levels instead of dropping one:
rec_tree <- recipe(~ platform, data = df) |> step_dummy(platform, one_hot = TRUE)

# Multiple columns at once -- step_dummy accepts a tidyselect expression
rec <- recipe(~ ., data = df) |> step_dummy(all_nominal_predictors())

# ── Encoding 2: Ordinal / Label Encoding ─────────────────────────────────────
# Use when: ordinal (has natural order) -- e.g. Low / Medium / High
# Do NOT use for nominal categories with tree-free models -- implies false ordering
order_map <- c(Low = 0, Medium = 1, High = 2, `Very High` = 3)
df <- df |> mutate(risk_level_enc = order_map[risk_level])

# Using R's ordered factor (remembers the order, R's direct analog to pd.Categorical(ordered=True))
df <- df |> mutate(risk_level = factor(risk_level,
  levels = c('Low', 'Medium', 'High', 'Very High'), ordered = TRUE
))
df <- df |> mutate(risk_level_enc = as.integer(risk_level) - 1)   # 0, 1, 2, 3 -- R factors are 1-indexed, subtract 1 to match

# ── Encoding 3: Frequency Encoding ──────────────────────────────────────────
# Use when: high cardinality, tree-based model, frequency itself is informative
# Replaces each category with how often it appears in the training set
freq_map <- df_train |> count(city) |> mutate(freq = n / sum(n)) |> select(city, freq)  # compute on train only
df <- df |> left_join(freq_map, by = "city") |> rename(city_freq = freq)

# ── Encoding 4: Target Encoding ──────────────────────────────────────────────
# Use when: high cardinality, strong relationship between category and target
# MAJOR LEAKAGE RISK -- must be done correctly with cross-validation
# Safe approach: compute on train set only, using out-of-fold means
library(rsample)

target_encode_cv <- function(df_train, df_test, col, target, n_splits = 5) {
  folds <- vfold_cv(df_train, v = n_splits)
  train_enc <- rep(NA_real_, nrow(df_train))
  global_mean <- mean(df_train[[target]], na.rm = TRUE)

  for (i in seq_len(nrow(folds))) {
    tr_idx  <- folds$splits[[i]] |> analysis() |> rownames() |> as.integer()
    val_idx <- folds$splits[[i]] |> assessment() |> rownames() |> as.integer()
    means <- df_train[tr_idx, ] |> group_by(.data[[col]]) |> summarise(m = mean(.data[[target]], na.rm = TRUE), .groups = "drop")
    lookup <- setNames(means$m, means[[col]])
    train_enc[val_idx] <- coalesce(lookup[df_train[[col]][val_idx]], global_mean)
  }

  # Apply to test: use full train set means
  train_means <- df_train |> group_by(.data[[col]]) |> summarise(m = mean(.data[[target]], na.rm = TRUE), .groups = "drop")
  lookup_full <- setNames(train_means$m, train_means[[col]])
  test_enc <- coalesce(lookup_full[df_test[[col]]], global_mean)
  list(train = train_enc, test = test_enc)
}

enc <- target_encode_cv(df_train, df_test, col = 'city', target = 'churn')
df_train$city_target <- enc$train
df_test$city_target  <- enc$test

**Encoding selection guide:**

| Situation | Encoding | Why |
| :--- | :--- | :--- |
| Nominal, low cardinality (<15), linear model | `step_dummy()` | No implied order |
| Ordinal (Low/Med/High) | Ordered factor / integer map | Order is meaningful |
| High cardinality (>50), tree model | Frequency | Avoids sparse columns |
| High cardinality, strong target signal | Target (with CV) | Most predictive — but leakage risk |

**Common mistakes:**
- Label-encoding a nominal column (e.g. `city`) for a linear model — implies `Paris > London > Tokyo` which is meaningless and distorts coefficients
- Forgetting that `step_dummy()` drops one level by default — this IS the `drop_first=True` behavior, so unlike pandas (where forgetting the argument is the bug), R's default is already correct for linear models; the trap runs the *other* direction here — remembering to pass `one_hot = TRUE` for tree models when you specifically don't want a level dropped
- Computing target encoding on the full training set without CV — your model sees the target in its features, causing massive overfitting

**A practical §1 footgun: train/test factor levels can mismatch.** Encoding
the two sets separately yields *different columns* whenever a category is
absent from one — the model then breaks at inference. recipes solves this
structurally: `step_dummy()` is **prepped on train and baked on test**, so the
column set it produces for test is locked to what it saw during `prep()` — a
category unseen in test simply gets a row of all zeros, no realignment step
needed. (Frequency encoding still has the sibling issue: categories unseen in
train map to `NA` via the `left_join` above — add `replace_na()`.)

In [ ]:
library(recipes)

train <- tibble::tibble(platform = c('ios','android','web'))
test  <- tibble::tibble(platform = c('ios','android'))        # 'web' never seen in test

cat('train dummies (manual, NO prep/bake -- shows the naive mismatch):\n')
print(names(fastDummies::dummy_cols(train, select_columns = "platform", remove_first_dummy = FALSE)))
cat('test  dummies (manual, naive):\n')
print(names(fastDummies::dummy_cols(test, select_columns = "platform", remove_first_dummy = FALSE)))
cat('  <- missing platform_web -- this is the SAME failure mode as pandas get_dummies()\n')

# Fix (the recipes way, preferred for ML): prep() locks the vocabulary to TRAIN, bake() applies it to test
rec <- recipe(~ platform, data = train) |> step_dummy(platform, one_hot = TRUE) |> prep()
tr  <- bake(rec, new_data = NULL)     # the training data, transformed
te  <- bake(rec, new_data = test)     # test data, using TRAIN's vocabulary -- platform_web column exists, filled with 0
cat('\naligned test (recipes prep/bake):\n')
print(names(te))

---
## §2 — Numerical Transformations

Scaling and transforming numeric features matters for distance-based models
(logistic regression, SVM, KNN, neural nets) but not for tree-based models
(XGBoost, Random Forest). Always `prep()` transforms on train, `bake()` test.

In [ ]:
library(recipes)

# ── Transform 1: Log transform -- for right-skewed distributions ──────────────
# Use when: revenue, income, counts -- anything with a long right tail
df <- df |> mutate(log_revenue = log1p(revenue))    # log(x+1): handles x=0 safely
# When NOT to use: symmetric distributions, negative values, already normal

# ── Transforms 2-5, expressed as ONE recipe (this IS the idiomatic R pattern --
# multiple step_*() calls compose into a single fit-on-train-only object,
# the direct structural analog to chaining several sklearn scalers) ──────────
rec <- recipe(~ age + income + score + revenue, data = X_train) |>
  step_normalize(age, income)                          # Transform 2: Standardization (Z-score), mean=0 sd=1 -- use for linear/SVM/KNN/NN
  # step_range(score)                                   # Transform 3: Min-Max Scaling to [0,1] -- use for neural nets, pixel values; sensitive to outliers
  # step_normalize(revenue)                              # would go here too -- shown split below for clarity on RobustScaler / YeoJohnson

rec_full <- recipe(~ age + income + score + revenue, data = X_train) |>
  step_normalize(age, income) |>                       # Transform 2: Standardization
  step_range(score, min = 0, max = 1) |>                # Transform 3: Min-Max Scaling
  step_YeoJohnson(revenue) |>                            # Transform 5: Yeo-Johnson -- auto power transform, handles zero/negative (Box-Cox needs step_BoxCox() and positive-only values)
  prep(training = X_train)

X_train_scaled <- bake(rec_full, new_data = NULL)
X_test_scaled  <- bake(rec_full, new_data = X_test)      # uses TRAIN's fitted mean/sd/range/lambda -- no separate .transform() call needed, bake() always means "apply the already-fit recipe"

# ── Transform 4: Robust Scaling -- uses median and IQR, not affected by outliers ──
# Use when: data has outliers you want to keep but not let dominate
# recipes has no single step_robust() -- compose it explicitly from its definition,
# OR reach for the `recipes`-compatible `themis`/`embed` extension packages if this
# is a frequent need; the explicit version makes exactly what "robust" means unmissable:
robust_scale <- function(x) (x - median(x, na.rm = TRUE)) / IQR(x, na.rm = TRUE)
X_train <- X_train |> mutate(revenue_robust = robust_scale(revenue))

# ── Quick check: plot before and after (ggplot2 instead of matplotlib) ───────
library(ggplot2)
library(patchwork)   # for side-by-side layout, ggplot2's analog to plt.subplots
p1 <- ggplot(df, aes(x = revenue))     + geom_histogram(bins = 50) + ggtitle("Raw")
p2 <- ggplot(df, aes(x = log_revenue)) + geom_histogram(bins = 50) + ggtitle("Log-transformed")
p1 + p2

**Transformation selection guide:**

| Situation | Transformation |
| :--- | :--- |
| Right-skewed (revenue, counts) | `log1p()` |
| Linear/logistic regression, SVM, KNN | `step_normalize()` |
| Neural network inputs | `step_range()` or `step_normalize()` |
| Data has significant outliers | manual median/IQR scaling (no single built-in step) |
| Want to auto-normalize any distribution | `step_YeoJohnson()` |
| Tree-based model (XGBoost, RF) | None required |

**Common mistakes:**
- Fitting the recipe on the full dataset before splitting — leaks test set distribution into the scaler; always `prep(training = X_train)`, never on the combined data
- Scaling the target variable when it isn't needed — scaling `y` for regression is usually unnecessary and complicates interpretation
- Using `log()` on columns with zeros — produces `-Inf`; always use `log1p()` for safety, same trap as NumPy's `np.log()`
- Calling `bake()` before `prep()`, or re-`prep()`-ing on test data by mistake — `prep()` is the ONLY place fitting happens; every subsequent `bake()` call, on train or test, reuses those fitted parameters

---
## §3 — Binning & Discretization

Convert a continuous variable into labeled groups. Useful when the
relationship with the target is non-linear and step-like, or when business
rules define meaningful ranges.

In [ ]:
library(dplyr)

# ── cut() -- equal-width bins, business-defined edges ────────────────────────
# Use when: bin edges come from domain knowledge or business rules
df <- df |> mutate(age_group = cut(
  age,
  breaks = c(0, 18, 35, 55, 100),
  labels = c('Under 18', '18-34', '35-54', '55+'),
  right = TRUE,             # intervals are (left, right] by default, same convention as pandas
  include.lowest = TRUE     # include the leftmost edge
))

# ── Equal-frequency (quantile) bins ──────────────────────────────────────────
# Use when: you want roughly equal-sized groups (e.g. quartiles)
# Important: compute quantiles on train set only
train_breaks <- quantile(df_train$revenue, probs = seq(0, 1, 0.25), na.rm = TRUE)
df_train <- df_train |> mutate(revenue_quartile = cut(
  revenue, breaks = train_breaks, labels = c('Q1','Q2','Q3','Q4'), include.lowest = TRUE
))

# Apply train bins to test (the breakpoints ARE the "fit" state -- store and reuse them,
# the direct R analog to pandas' retbins=True)
df_test <- df_test |> mutate(revenue_quartile = cut(
  revenue, breaks = train_breaks, labels = c('Q1','Q2','Q3','Q4'), include.lowest = TRUE
))
# Equivalently, as a recipe step: recipe(...) |> step_discretize(revenue, num_breaks = 4) |> prep(training = df_train)
# step_discretize() handles the fit-on-train/apply-to-test split automatically via prep()/bake()

# ── Custom business-logic bins ────────────────────────────────────────────────
# Use when: business definitions dictate the thresholds
# Vectorized via case_when() -- R's direct analog to np.select(), preferred over rowwise logic
df <- df |> mutate(user_segment = case_when(
  revenue >= 1000 & tenure_days >= 365 ~ 'VIP',
  revenue >= 500                       ~ 'Premium',
  revenue > 0                          ~ 'Standard',
  .default = 'Inactive'
))

# ── Check bin distribution after cutting ─────────────────────────────────────
table(df$age_group)

**`cut()` with fixed breaks vs quantile breaks:**

| | Fixed `breaks=` | `quantile()`-derived `breaks=` |
| :--- | :--- | :--- |
| Bin width | Equal width | Equal frequency |
| Group sizes | Unequal (depends on distribution) | Roughly equal |
| Edges from | You specify | Computed from data |
| Use when | Domain-defined thresholds | Want balanced groups |
| Leakage risk | Low | High — compute on train only |

**Common mistakes:**
- Computing quantile breakpoints on the full dataset — bin edges depend on the distribution, which leaks test set info into the model, identical trap to pandas' `qcut`
- Forgetting `include.lowest = TRUE` — the lowest value falls outside all bins and becomes `NA`
- Using `cut()` labels as a plain factor directly in a linear model without an explicit encoding step — `lm()`/`glm()` actually handle unordered factors correctly via internal dummy coding, but for `recipes`-based workflows, run `step_dummy()` on the binned column afterward to be explicit about the encoding

---
## §4 — Date & Time Features

Raw timestamps are not useful to most models. Extract meaningful components —
and encode cyclical features correctly so models understand that hour 23 is
close to hour 0.

In [ ]:
library(lubridate)
library(dplyr)

df <- df |> mutate(ts = ymd_hms(ts))

# ── Basic extractions ────────────────────────────────────────────────────────
df <- df |> mutate(
  year         = year(ts),
  month        = month(ts),                      # 1-12
  day          = day(ts),                         # 1-31
  hour         = hour(ts),                         # 0-23
  day_of_week  = wday(ts, week_start = 1) - 1,     # 0=Mon, 6=Sun -- match pandas' dayofweek convention explicitly (see Tidyverse ref SS8)
  is_weekend   = as.integer(wday(ts, week_start = 1) - 1 >= 5),
  quarter      = quarter(ts),                       # 1-4
  week_of_year = isoweek(ts)
)

# ── Cyclical encoding -- critical for hour, month, day_of_week ───────────────
# Problem: raw hour 23 and hour 0 are far apart numerically but close in time
# Solution: encode as (sin, cos) pair -- places similar times close in 2D space
cyclical_encode <- function(x, max_val) {
  list(sin = sin(2 * pi * x / max_val), cos = cos(2 * pi * x / max_val))
}

hour_enc  <- cyclical_encode(df$hour, 24)
month_enc <- cyclical_encode(df$month, 12)
dow_enc   <- cyclical_encode(df$day_of_week, 7)
df <- df |> mutate(
  hour_sin = hour_enc$sin, hour_cos = hour_enc$cos,
  month_sin = month_enc$sin, month_cos = month_enc$cos,
  dow_sin = dow_enc$sin, dow_cos = dow_enc$cos
)
# Equivalently via recipes: step_date(ts, features = c("dow","month")) |> step_harmonic(...)
# handles the extraction + cyclical encoding together for recipe-based pipelines

# ── Elapsed time features ────────────────────────────────────────────────────
reference_date <- ymd('2020-01-01')
df <- df |> mutate(
  days_since_ref    = as.numeric(as_date(ts) - reference_date, units = "days"),
  days_since_signup = as.numeric(as_date(ts) - as_date(signup_date), units = "days"),
  days_to_expiry    = as.numeric(as_date(expiry_date) - as_date(ts), units = "days")
)

# ── Time-since-last-event (per user) ─────────────────────────────────────────
df <- df |> arrange(user_id, ts) |> group_by(user_id) |> mutate(
  prev_ts          = lag(ts),
  hours_since_last = as.numeric(difftime(ts, prev_ts, units = "hours"))
) |> ungroup()

# ── Business day features ─────────────────────────────────────────────────────
# R has no single bundled "USFederalHolidayCalendar" the way pandas does;
# timeDate provides the standard US exchange holiday calendars directly
library(timeDate)
holidays <- as.Date(holidayNYSE(year = unique(year(df$ts))))   # NYSE calendar is the closest standard US business-holiday set
df <- df |> mutate(
  is_holiday      = as.integer(as_date(ts) %in% holidays),
  is_business_day = as.integer(!(wday(ts, week_start = 1) %in% c(6, 7)) & !as.logical(is_holiday))
)

**When to use cyclical encoding:**

| Feature | Range | Use cyclical? |
| :--- | :--- | :--- |
| Hour of day | 0-23 | Yes — 23 is close to 0 |
| Day of week | 0-6 | Yes — Sunday is close to Monday |
| Month | 1-12 | Yes — December is close to January |
| Year | 2020-2024 | No — not cyclical |
| Days since event | 0-infinity | No — not cyclical |

**Common mistakes:**
- Using raw hour as a numeric feature — model sees hour 1 and hour 23 as far apart; use sin/cos encoding instead
- Computing `days_since_signup`-style lag features without `arrange()`-ing by user and date first — `lag()` operates on row order, not time order
- Extracting too many date features — correlated features (month + quarter + season) add noise; pick the most granular
- Reaching for a US-Python-package-specific holiday calendar name out of habit — `timeDate::holidayNYSE()` (or the broader `holidayNERC()`/custom calendars) is R's standard source for US business-day calendars; there is no exact 1:1 package name match, so verify the specific holiday set matches your business need

---
## §5 — Lag & Window Features

Use past values to predict future values. The most powerful feature type for
time-series and sequential data — and the easiest place to introduce
lookahead leakage.

In [ ]:
library(dplyr)
library(slider)

# Always sort before creating lag or window features
df <- df |> arrange(user_id, date)

# ── Lag features -- past values of the target or other columns ────────────────
# LAG(revenue, 1): revenue from previous period
df <- df |> group_by(user_id) |> mutate(
  rev_lag1  = lag(revenue, 1),    # 1 period ago
  rev_lag7  = lag(revenue, 7),    # 7 periods ago
  rev_lag30 = lag(revenue, 30)    # 30 periods ago
) |> ungroup()

# ── Rolling window features -- statistics over the last N rows ─────────────────
# slide_dbl(..., .before = n-1) on the LAGGED series prevents using the current row
# -- the lag(revenue, 1) wrapping is the R equivalent of pandas' shift(1).rolling(7)
df <- df |> group_by(user_id) |> mutate(
  rev_rolling_mean_7 = slide_dbl(lag(revenue, 1), mean, .before = 6,  na.rm = TRUE, .complete = FALSE),
  rev_rolling_std_7  = slide_dbl(lag(revenue, 1), sd,   .before = 6,  na.rm = TRUE, .complete = FALSE),
  rev_rolling_max_30 = slide_dbl(lag(revenue, 1), max,  .before = 29, na.rm = TRUE, .complete = FALSE)
) |> ungroup()

# ── Expanding window -- cumulative from the start of the series ───────────────
# Useful for lifetime statistics (total purchases ever)
df <- df |> group_by(user_id) |> mutate(
  cumulative_rev = slide_dbl(lag(revenue, 1), sum, .before = Inf, na.rm = TRUE)   # sum of all past revenue, not including current
) |> ungroup()

# ── Delta features -- change from previous period ─────────────────────────────
df <- df |> mutate(
  rev_delta      = revenue - rev_lag1,                                     # absolute change
  rev_pct_change = rev_delta / na_if(rev_lag1, 0)                          # % change, na_if() is R's .replace(0, NA) equivalent
)

# ── Safe train/test split for time series ────────────────────────────────────
# NEVER use random split -- it causes lookahead leakage
# Use a cutoff date instead -- rsample::initial_time_split() is the purpose-built
# tidymodels function for exactly this (orders by row, splits at a proportion --
# pass already date-sorted data, or use a manual cutoff filter as shown for full control)
cutoff <- ymd('2024-01-01')
train <- df |> filter(date <  cutoff)
test  <- df |> filter(date >= cutoff)
# Verify: test features should only use data from before the cutoff
# Lag features for test rows should look back into train rows -- this is correct
# Rolling features for test rows that include post-cutoff data -- this is leakage

**Leakage anatomy for lag features:**

```
Timeline: ... | day 5 | day 6 | day 7 (target) |

lag1 = day 6 revenue                            <- OK: past data
lag7 = day 0 revenue                             <- OK: past data
rolling_mean WITHOUT lag(1) wrap = mean(day 5, 6, 7)  <- LEAKAGE: includes target day
rolling_mean WITH lag(1) wrap    = mean(day 4, 5, 6)  <- OK: excludes target day
```

**Common mistakes:**
- Not wrapping the series in `lag(x, 1)` before `slide_dbl()` — the rolling window includes the current row, which is the target, the exact R analog to forgetting `.shift(1)` before `.rolling()`
- Using a random `initial_split()` for time series — future rows leak into training via lag features; use `initial_time_split()` or a manual date cutoff instead
- Not `arrange()`-ing by date within each group before `lag()` — shift operates on row position, not time order
- Forgetting `.complete = FALSE` in `slide_dbl()` — without it, the first N-1 rows of each group become `NA` *and stay that way regardless of available data*; `.complete = FALSE` is `slider`'s analog to pandas' `min_periods=1`, computing a shorter window rather than discarding the row

---
## §6 — Interaction & Ratio Features

Combining two or more columns can capture relationships that neither column
captures alone. Most valuable when domain knowledge suggests the combination
is meaningful.

In [ ]:
library(dplyr)
library(recipes)

# ── Ratio features -- normalize one column by another ─────────────────────────
# Use when: the absolute value is less meaningful than the relative value
df <- df |> mutate(
  ctr             = clicks / na_if(impressions, 0),       # click-through rate
  conversion_rate = conversions / na_if(clicks, 0),
  rev_per_session = revenue / na_if(sessions, 0),
  arpu            = revenue / na_if(users, 0)              # avg revenue per user
)

# ── Difference features -- gap between two related values ─────────────────────
df <- df |> mutate(
  price_vs_avg    = price - category_avg_price,            # above/below category mean
  score_delta     = current_score - baseline_score,
  days_to_renewal = as.numeric(as_date(renewal_date) - as_date(today))
)

# ── Product (multiplication) -- interaction between two features ───────────────
# Use when: both features together affect the outcome (synergy effect)
df <- df |> mutate(
  tenure_x_spend = tenure_days * total_spend,    # loyalty x value
  age_x_income   = age * income                   # demographic interaction
)

# ── Polynomial features -- automated interaction and power terms ───────────────
# Use for linear models when you suspect non-linear relationships
rec <- recipe(~ age + income + tenure, data = X_train) |>
  step_poly(age, income, tenure, degree = 2) |>
  step_interact(terms = ~ age:income + age:tenure + income:tenure) |>
  prep()
X_poly <- bake(rec, new_data = NULL)
# Produces: age, income, tenure, their degree-2 polynomial terms, and pairwise
# interactions -- the recipes equivalent of sklearn's PolynomialFeatures(degree=2)
# Warning: features grow as O(n^2) -- only use with a small number of base features

# ── Group-relative features -- how does this row compare to its group? ─────────
df <- df |> group_by(region) |> mutate(
  rev_vs_region_avg  = revenue - mean(revenue, na.rm = TRUE),
  rev_pct_of_region   = revenue / sum(revenue, na.rm = TRUE),
  rev_rank_in_region  = dense_rank(desc(revenue))
) |> ungroup()

**Common mistakes:**
- Dividing without handling zeros — `clicks / impressions` when `impressions = 0` produces `Inf` or `NaN`; always `na_if(denominator, 0)` first, R's direct analog to pandas' `.replace(0, np.nan)`
- Creating all possible interactions automatically without domain guidance — most interactions are noise; domain-motivated features outperform automated `step_poly()`/`step_interact()` features
- Not checking for multicollinearity after adding ratio features — `rev_per_user`, `total_rev`, and `user_count` are linearly dependent; use `car::vif()` or a correlation matrix (`cor()`) to diagnose

---
## §7 — Text as Features

For tabular ML problems with one or two text columns — not full NLP, just
enough to extract signal from free text. `textrecipes` extends the
`recipes` framework with text-specific `step_*()` functions, the direct
tidymodels analog to scikit-learn's `feature_extraction.text` module.

In [ ]:
library(stringr)
library(dplyr)
library(textrecipes)
library(recipes)

# ── Simple text features -- no vectorization needed ───────────────────────────
df <- df |> mutate(
  desc_length     = str_length(description),                                          # character count
  desc_word_count = str_count(description, "\\S+"),                                     # word count
  has_promo       = as.integer(str_detect(str_to_lower(description), "sale|discount|free")),
  exclamation_ct  = str_count(description, "!"),
  has_url         = as.integer(str_detect(description, "https?://"))
)

# ── TF-IDF -- term frequency-inverse document frequency ───────────────────────
# Use when: text column has meaningful vocabulary and medium cardinality
# Fit (prep) on train, bake both train and test -- textrecipes folds tokenization,
# stopword removal, and TF-IDF weighting into ordinary recipe steps
rec <- recipe(~ description, data = X_train) |>
  step_tokenize(description) |>
  step_stopwords(description) |>                          # removes 'the', 'and', 'is', etc.
  step_ngram(description, num_tokens = 2, min_num_tokens = 1) |>  # include bigrams: 'fast shipping' as one feature
  step_tokenfilter(description, max_tokens = 100, min_times = 5) |>  # keep top-100 terms, ignore rare ones
  step_tfidf(description) |>
  prep(training = X_train)

tfidf_train_df <- bake(rec, new_data = NULL)    # already includes the new tfidf_* columns
tfidf_test_df  <- bake(rec, new_data = X_test)  # uses TRAIN's fitted vocabulary and IDF weights
X_train <- bind_cols(X_train, tfidf_train_df |> select(starts_with("tfidf_")))

# ── Bag of Words (raw counts instead of TF-IDF weights) ───────────────────────
# Use when: raw term counts are more useful than TF-IDF weights
# Less common than TF-IDF but useful when document length is consistent
rec_bow <- recipe(~ description, data = X_train) |>
  step_tokenize(description) |>
  step_stopwords(description) |>
  step_tokenfilter(description, max_tokens = 50) |>
  step_tf(description) |>                                  # step_tf() instead of step_tfidf() -- raw counts, no IDF weighting
  prep(training = X_train)
bow_train <- bake(rec_bow, new_data = NULL)

# ── Category from text -- keyword-based labeling ───────────────────────────────
# Use when: a few keywords cleanly define the category
keyword_map <- list(
  electronics = c('phone', 'laptop', 'tablet', 'camera'),
  clothing    = c('shirt', 'pants', 'dress', 'shoes'),
  food        = c('snack', 'drink', 'meal', 'coffee')
)
classify_by_keywords <- function(text, keyword_map) {
  text <- str_to_lower(as.character(text))
  for (category in names(keyword_map)) {
    if (any(str_detect(text, keyword_map[[category]]))) return(category)
  }
  'other'
}
df <- df |> mutate(product_category = purrr::map_chr(description, classify_by_keywords, keyword_map = keyword_map))

**Common mistakes:**
- `prep()`-ing the text recipe on the full dataset — vocabulary and IDF weights leak test distribution; always `prep(training = X_train)`
- Not filling `NA` before tokenizing — `step_tokenize()` errors on `NA` values; use `mutate(description = replace_na(description, ""))` first, identical trap to pandas' `TfidfVectorizer` on un-filled NaN
- Using too many `max_tokens` — 100-500 is usually enough; more features add noise and memory pressure

---
## §8 — Leakage & Validation

Feature leakage is when information from the future (or from the target
itself) is encoded into features used to train the model. It produces
unrealistically high training performance and catastrophic failure in
production.

In [ ]:
library(dplyr)
library(rsample)

# ── Type 1: Target leakage -- feature is computed using the target ─────────────
# Example: predicting loan default, and a feature is 'days_past_due'
# days_past_due is 0 for non-defaulters and >0 for defaulters -- it IS the target

# Detection: check correlation of each feature with the target
correlations <- sapply(X_train, function(col) if (is.numeric(col)) abs(cor(col, y_train, use = "complete.obs")) else NA)
print(sort(correlations, decreasing = TRUE, na.last = NA)[1:10])
# Suspicious: correlation > 0.8 with the target -> investigate that feature

# Detection: check if any feature is derived from post-event data
# Ask for each feature: "Would I know this value at prediction time?"
# e.g. 'refund_amount' for a churn model -- you only know the refund AFTER churn happens

# ── Type 2: Temporal leakage -- using future data for a past prediction ────────
# Wrong: random split on time-series data
split_wrong <- initial_split(df, prop = 0.8)   # rsample's default initial_split() IS a random split
# This mixes future dates into training -- lag features for training rows
# can look up rows that are chronologically later

# Correct: time-based split -- rsample's purpose-built function for exactly this
df_sorted <- df |> arrange(date)
split_correct <- initial_time_split(df_sorted, prop = 0.8)   # takes the FIRST 80% of rows as train, by position
train <- training(split_correct)
test  <- testing(split_correct)
# Equivalently, an explicit cutoff for full control (same result, more transparent):
cutoff <- quantile(df$date, 0.8)   # or a fixed business date
train  <- df |> filter(date <= cutoff)
test   <- df |> filter(date >  cutoff)

# ── Type 3: Group leakage -- same entity in train and test ─────────────────────
# Example: predicting customer LTV; same customer_id in both train and test
# The model memorizes individual customers rather than learning patterns

# Correct: group-aware split (all rows of one entity go to one split) --
# rsample's group_initial_split() is the direct analog to GroupShuffleSplit
split_group <- group_initial_split(df, group = user_id, prop = 0.8)
X_train <- training(split_group)
X_test  <- testing(split_group)

# ── Leakage diagnosis checklist ──────────────────────────────────────────────
# 1. Model accuracy is suspiciously high (>95% on a hard problem)?
# 2. Performance drops dramatically from validation to production?
# 3. Feature importance shows an unexpected column dominating?
# 4. Features computed after the prediction point (post-event data)?
# 5. Rolling/window features that didn't lag(1) before slide_dbl()?
# 6. Recipes prep()-ed on the full dataset instead of train only?

# ── Safe preprocessing pipeline -- prep on train, bake both ───────────────
library(recipes)

rec <- recipe(~ ., data = X_train) |>
  step_impute_median(all_numeric_predictors()) |>
  step_normalize(all_numeric_predictors()) |>
  prep(training = X_train)               # fit on train

X_train_processed <- bake(rec, new_data = NULL)   # apply to train
X_test_processed  <- bake(rec, new_data = X_test)  # apply to test -- no re-fitting, ever

**Leakage types and fixes:**

| Type | Signal | Fix |
| :--- | :--- | :--- |
| Target leakage | Feature correlates >0.8 with target; comes from post-event data | Remove the feature; ask "would I know this at prediction time?" |
| Temporal leakage | Random split on time-series; rolling features without `lag(1)` | `initial_time_split()` / cutoff filter; always wrap in `lag(1)` before `slide_dbl()` |
| Group leakage | Same entity ID in train and test | `group_initial_split()` or entity-level split |
| Preprocessing leakage | Recipe `prep()`-ed on full data before split | `prep(training = X_train)` only, never on combined data |
| Target encoding leakage | Category means computed without CV | Use out-of-fold CV target encoding (see §1) |

**Common mistakes:**
- Treating suspiciously high accuracy as a win — if a churn model hits 99% AUC, check for leakage before celebrating
- Filling missing values before splitting — the imputed value (e.g. median) is computed from the full dataset, leaking test statistics into train; this is exactly what `step_impute_median()` inside a `prep(training = X_train)`-scoped recipe prevents structurally
- Manually computing scaling/imputation statistics outside a `recipe()` and applying them by hand to both sets — the `recipe()`/`prep()`/`bake()` pattern enforces correct fit-on-train behavior the same way `sklearn.Pipeline` does; bypassing it reintroduces the exact risk it exists to close off

---
## Decision Guide

```
Encoding a categorical column?
+-- Nominal, <15 unique values, linear model    -> step_dummy() (one level dropped by default)  (§1)
+-- Ordinal (Low/Med/High)                      -> ordered factor + as.integer()                (§1)
+-- High cardinality, tree model                -> frequency encoding (count() + left_join)      (§1)
\-- High cardinality, strong target signal      -> target encoding with CV (vfold_cv)            (§1)

Transforming a numeric column?
+-- Right-skewed (revenue, counts)              -> log1p()                                       (§2)
+-- Linear model, SVM, KNN, NN                  -> step_normalize() (fit on train only)          (§2)
+-- Has significant outliers                    -> manual median/IQR scaling                     (§2)
\-- Want auto-normalization                     -> step_YeoJohnson()                              (§2)

Binning a continuous variable?
+-- Business-defined thresholds                 -> cut(breaks = ...)                              (§3)
+-- Equal-sized groups                          -> quantile() breaks on train only, then cut()    (§3)
\-- Multi-condition business logic              -> case_when()                                    (§3)

Extracting from a datetime column?
+-- Cyclical component (hour, month, weekday)   -> sin/cos encoding                              (§4)
+-- Time elapsed since an event                 -> as.numeric(as_date(ts) - ref_date)             (§4)
\-- Time since last event per user              -> group_by() |> lag() -> difftime in hours       (§4)

Creating lag/window features?
+-- Previous value                              -> group_by() |> lag()                            (§5)
+-- Rolling mean/std over last N rows           -> slide_dbl(lag(x,1), fn, .before=N-1)           (§5)
+-- Cumulative sum from start of series         -> slide_dbl(lag(x,1), sum, .before=Inf)          (§5)
\-- Train/test split                            -> initial_time_split() / date cutoff, NOT random (§5)

Combining two columns?
+-- One normalizes the other (CTR, ARPU)        -> ratio feature (na_if() the denominator)       (§6)
+-- Gap from a reference (vs group mean)        -> difference feature                             (§6)
\-- Both features interact                      -> product feature / step_interact()              (§6)

Text column in a tabular dataset?
+-- Simple signal (length, keywords, counts)    -> str_length(), str_detect()                    (§7)
\-- Richer vocabulary signal                    -> textrecipes::step_tfidf() (fit on train only)  (§7)

Suspecting leakage?
+-- Model accuracy is unrealistically high      -> check feature correlations with target         (§8)
+-- Time-series with random split               -> switch to initial_time_split()                 (§8)
+-- Same user in train and test                 -> group_initial_split()                          (§8)
\-- Preprocessing fit on full data              -> use a recipe(), prep(training=X_train) only    (§8)
```

**§8 capstone: the full `recipe()`, the direct structural analog to
`ColumnTransformer`.** §8 shows individual steps in isolation, but real
feature sets mix column types in one object. Unlike scikit-learn (where
`ColumnTransformer` is a *separate* wrapper needed specifically because
individual transformers are column-blind), `recipes::recipe()` **already**
routes each `step_*()` to the right columns via `tidyselect` — there is no
extra wrapping layer to learn. `prep()` fits on train; `bake()` applies
to test.

In [ ]:
library(recipes)

X_train <- tibble::tibble(
  age      = c(25, 40, NA, 33),
  platform = c('ios','web','ios','android')
)

rec <- recipe(~ age + platform, data = X_train) |>
  step_impute_median(age) |>                              # numeric pipeline, step 1: impute
  step_normalize(age) |>                                   # numeric pipeline, step 2: scale
  step_dummy(platform, one_hot = TRUE) |>                  # categorical pipeline: one-hot encode
  prep(training = X_train)                                  # fit on TRAIN only; use bake(rec, new_data = X_test) on test

out <- bake(rec, new_data = NULL)
cat('output shape :', paste(dim(out), collapse = " x "), '\n')
cat('feature names:', paste(names(out), collapse = ", "), '\n')